# Hybrid billing-price prediction

This notebook predicts a future **base billing amount** from historical rent/additional-rent lines, then applies the tax and billing formulas from `Tax_Formulas_Expanded.md` to produce an auditable final price.

The input contract is the frontend payload requested in the specification:
`year`, `month`, `amount`, `cgst`, `sgst`, `billing_charge`, `area_bill_per`, `area`, `target_year`, and `target_month`.

> Important production requirement: the formula sheet says that general, sewerage, water, street, MECESS, Tree Cess, WBT, SBT, and EGCESS rates come from master records. The provided CSV does not contain all of those master rates, so the notebook keeps them in one explicit configuration dictionary. Replace the worked-example defaults before production use.

## Execution plan

1. **Load and audit data**: read only the CSV columns needed by the pipeline, coerce numeric fields, create a billing period index, and preserve the source formula markdown for traceability.
2. **Preprocess and classify lines**: normalize area and billing-frequency fields, classify rent/additional-rent lines separately from tax lines, and aggregate repeated customer/category/month records.
3. **Create a forecasting target**: for each customer and base-line category, pair the current historical observation with the next available observation. The model learns the future base amount without using the future amount as a feature.
4. **Feature engineering**: use the present base amount, present GST amounts, area, calendar seasonality, forecast horizon, amount-per-area, billing frequency, and base-line category. The target is modeled as `log1p(amount)` to reduce the effect of extreme billing outliers.
5. **Model and time-based validation**: fit an XGBoost gradient-boosted regression model on `log1p(base_amount)` before the validation cutoff, then evaluate on later target periods. Compare against a persistence baseline and report both raw-currency and log-space R².
6. **Formula integration**: calculate AM, LV, GRVP, NRVP, GRVS, and NRVS in the exact order specified. Apply the correct pre-tax or post-tax schedule for the target month, use the configured NRV constant, infer CGST/SGST rates from the present bill, and apply area-period charge scaling.
7. **Final audit and handoff**: show the validation matrix, run the worked example from the formula sheet as a unit test, refit the production model on all historical pairs, and expose one frontend-ready prediction function.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

DATA_PATH = Path('C:/Users/kumar/Desktop/dataset/billing_training_dataset.csv')
FORMULA_PATH = Path('C:/Users/kumar/Downloads/Tax_Formulas_Expanded.md')
RANDOM_STATE = 42
VALIDATION_START_YEAR = 2025
VALIDATION_START_MONTH = 1
EPSILON = 1e-12

# These values are the worked-example rates printed in the formula markdown.
# Replace them with the applicable master-record rates before production use.
TAX_RATES = {
    'general': 0.15,
    'sewerage': 0.08,
    'water': 0.06,
    'street': 0.02,
    'mecess': 0.02,
    'tree_cess': 0.01,
    'wbt': 0.03,
    'sbt': 0.02,
    'egcess': 0.01,
}

NRV_CONSTANTS = {'mbpt': 0.837, 'other': 0.792}
BILLING_PERIOD_MONTHS = {'monthly': 1.0, 'half_yearly': 6.0, 'yearly': 12.0}
PRE_TAX_MONTHS = {4, 10}
POST_TAX_MONTHS = {3, 9}

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Training dataset not found: {DATA_PATH}')
if not FORMULA_PATH.exists():
    raise FileNotFoundError(f'Formula sheet not found: {FORMULA_PATH}')


In [ ]:
# Keep the source specification available in the notebook for auditability.
formula_text = FORMULA_PATH.read_text(encoding='utf-8', errors='replace')
formula_headings = [line for line in formula_text.splitlines() if line.startswith('#')]
print('Loaded formula specification:', FORMULA_PATH)
print('Formula sections:')
print('\n'.join(formula_headings))


In [ ]:
# Read only the fields required by the model and the formula audit.
REQUIRED_COLUMNS = [
    'src_customerid', 'src_billyearmonth', 'src_billchargeid', 'src_amount',
    'src_sgst', 'src_cgst', 'src_area_in_sqm', 'src_unit', 'src_month', 'src_year',
    'dim_bill_head_name', 'dim_bill_head_category',
    'dim_property_bill_periodicity', 'dim_latest_area', 'dim_letout_billable_area',
]

def load_training_data(path: Path) -> pd.DataFrame:
    header = pd.read_csv(path, nrows=0)
    available = [column for column in REQUIRED_COLUMNS if column in header.columns]
    missing = sorted(set(REQUIRED_COLUMNS) - set(available))
    if missing:
        raise ValueError(f'The CSV is missing required columns: {missing}')
    data = pd.read_csv(path, usecols=available, low_memory=False)
    numeric_columns = [
        'src_customerid', 'src_billyearmonth', 'src_billchargeid', 'src_amount',
        'src_sgst', 'src_cgst', 'src_area_in_sqm', 'src_unit', 'src_month', 'src_year',
        'dim_latest_area', 'dim_letout_billable_area',
    ]
    for column in numeric_columns:
        if column in data.columns:
            data[column] = pd.to_numeric(data[column], errors='coerce')
    data = data.dropna(subset=['src_customerid', 'src_billyearmonth', 'src_amount']).copy()
    data['src_billyearmonth'] = data['src_billyearmonth'].astype(int)
    data['period_index'] = data['src_billyearmonth'].astype(int)
    data['period_year'] = data['period_index'] // 100
    data['period_month'] = data['period_index'] % 100
    data = data[data['period_month'].between(1, 12)].copy()
    data['period_index'] = data['period_year'] * 12 + data['period_month']
    data['amount'] = data['src_amount'].astype(float)
    data['cgst'] = data['src_cgst'].fillna(0).astype(float)
    data['sgst'] = data['src_sgst'].fillna(0).astype(float)
    area_sources = [
        data['src_area_in_sqm'], data['dim_latest_area'], data['dim_letout_billable_area']
    ]
    data['area'] = area_sources[0]
    for area_source in area_sources[1:]:
        data['area'] = data['area'].combine_first(area_source)
    data['area'] = pd.to_numeric(data['area'], errors='coerce').fillna(0).clip(lower=0)
    return data

billing = load_training_data(DATA_PATH)
print(f'Loaded {len(billing):,} rows and {billing["src_customerid"].nunique():,} customers')
print(f'Historical period range: {billing["src_billyearmonth"].min()} to {billing["src_billyearmonth"].max()}')


In [ ]:
def normalize_frequency(value) -> str:
    text = str(value).strip().lower().replace('-', '_').replace(' ', '_')
    aliases = {
        'monthly': 'monthly',
        'month': 'monthly',
        'yearly': 'yearly',
        'annual': 'yearly',
        'annually': 'yearly',
        'half_yearly': 'half_yearly',
        'halfyearly': 'half_yearly',
        'half_annually': 'half_yearly',
        'semi_annual': 'half_yearly',
        'semiannual': 'half_yearly',
    }
    return aliases.get(text, 'monthly')

def classify_line_category(data: pd.DataFrame) -> pd.Series:
    category = data['dim_bill_head_category'].fillna('').astype(str)
    name = data['dim_bill_head_name'].fillna('').astype(str)
    text = (category + ' ' + name).str.lower()
    conditions = [
        text.str.contains('mecess|education cess', regex=True),
        text.str.contains('tree cess|treecess', regex=True),
        text.str.contains('water benefit|wbt', regex=True),
        text.str.contains('sewerage benefit|sbt', regex=True),
        text.str.contains('employee guarantee|egcess', regex=True),
        text.str.contains('street tax', regex=True),
        text.str.contains(r'prop\.tax|property tax', regex=True),
        text.str.contains('rent|licence|license', regex=True),
        text.str.contains('7a|additional rent', regex=True),
    ]
    choices = [
        'mecess', 'tree_cess', 'wbt', 'sbt', 'egcess', 'street_tax',
        'property_tax', 'rent', 'additional_rent',
    ]
    return pd.Series(np.select(conditions, choices, default='other'), index=data.index)

billing['line_category'] = classify_line_category(billing)
billing['billing_frequency'] = billing['dim_property_bill_periodicity'].map(normalize_frequency)

print('Line categories:')
display(billing['line_category'].value_counts().rename_axis('line_category').to_frame('rows'))
print('Billing frequencies:')
display(billing['billing_frequency'].value_counts().rename_axis('billing_frequency').to_frame('rows'))

# The model forecasts base rent components. Tax lines remain available for audit but are not used as base targets.
BASE_CATEGORIES = {'rent', 'additional_rent'}
base_history = billing[billing['line_category'].isin(BASE_CATEGORIES)].copy()
base_history = base_history[base_history['amount'].notna() & (base_history['amount'] > 0)].copy()
if base_history.empty:
    raise ValueError('No rent/additional-rent history was found after classification.')
TRAINING_AREA_MEDIAN = float(base_history['area'].replace(0, np.nan).median())
if not np.isfinite(TRAINING_AREA_MEDIAN):
    TRAINING_AREA_MEDIAN = 0.0

def stable_mode(values, default='monthly'):
    values = values.dropna().astype(str)
    return values.mode().iloc[0] if not values.empty else default

# Multiple source rows can represent the same customer/category/month. Aggregate them before creating lags.
monthly_base = (
    base_history.groupby(['src_customerid', 'line_category', 'period_index'], as_index=False, dropna=False)
    .agg(
        present_amount=('amount', 'sum'),
        present_cgst=('cgst', 'sum'),
        present_sgst=('sgst', 'sum'),
        present_area=('area', 'median'),
        billing_frequency=('billing_frequency', stable_mode),
    )
)
monthly_base = monthly_base.sort_values(['src_customerid', 'line_category', 'period_index']).reset_index(drop=True)
grouped = monthly_base.groupby(['src_customerid', 'line_category'], sort=False)
monthly_base['target_amount'] = grouped['present_amount'].shift(-1)
monthly_base['target_period_index'] = grouped['period_index'].shift(-1)
monthly_base['horizon_months'] = monthly_base['target_period_index'] - monthly_base['period_index']
monthly_base = monthly_base[monthly_base['target_amount'].notna()].copy()
monthly_base = monthly_base[monthly_base['horizon_months'] > 0].copy()
monthly_base['present_year'] = ((monthly_base['period_index'] - 1) // 12).astype(int)
monthly_base['present_month'] = ((monthly_base['period_index'] - 1) % 12 + 1).astype(int)
monthly_base['target_year'] = ((monthly_base['target_period_index'] - 1) // 12).astype(int)
monthly_base['target_month'] = ((monthly_base['target_period_index'] - 1) % 12 + 1).astype(int)
monthly_base['present_area'] = monthly_base['present_area'].fillna(TRAINING_AREA_MEDIAN).clip(lower=0)
monthly_base['present_amount_per_area'] = monthly_base['present_amount'] / monthly_base['present_area'].replace(0, np.nan)
monthly_base['present_amount_per_area'] = monthly_base['present_amount_per_area'].replace([np.inf, -np.inf], np.nan).fillna(0)
monthly_base['present_log_amount'] = np.log1p(monthly_base['present_amount'].clip(lower=0))
monthly_base['target_amount'] = pd.to_numeric(monthly_base['target_amount'], errors='coerce')
monthly_base = monthly_base[monthly_base['target_amount'] > 0].copy()

FEATURE_COLUMNS = [
    'present_amount', 'present_cgst', 'present_sgst', 'present_area',
    'present_year', 'present_month', 'target_year', 'target_month',
    'horizon_months', 'present_amount_per_area', 'present_log_amount',
    'billing_frequency', 'line_category',
]
NUMERIC_FEATURES = [column for column in FEATURE_COLUMNS if column not in {'billing_frequency', 'line_category'}]
CATEGORICAL_FEATURES = ['billing_frequency', 'line_category']
print(f'Created {len(monthly_base):,} supervised base-amount pairs')


In [ ]:
def make_one_hot_encoder():
    # sklearn renamed this argument; keep the notebook compatible with both APIs.
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

def make_model_pipeline() -> Pipeline:
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
    ])
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('one_hot', make_one_hot_encoder()),
    ])
    preprocessor = ColumnTransformer([
        ('numeric', numeric_pipeline, NUMERIC_FEATURES),
        ('categorical', categorical_pipeline, CATEGORICAL_FEATURES),
    ], remainder='drop')
    regressor = GradientBoostingRegressor(
        n_estimators=120,
        learning_rate=0.08,
        max_depth=2,
        min_samples_leaf=30,
        loss='huber',
        random_state=RANDOM_STATE,
    )
    return Pipeline([('preprocessor', preprocessor), ('regressor', regressor)])

def period_index(year, month):
    return int(year) * 12 + int(month)

validation_cutoff = period_index(VALIDATION_START_YEAR, VALIDATION_START_MONTH)
train_pairs = monthly_base[monthly_base['target_period_index'] < validation_cutoff].copy()
test_pairs = monthly_base[monthly_base['target_period_index'] >= validation_cutoff].copy()
if train_pairs.empty or test_pairs.empty:
    raise ValueError('The time-based train/test split is empty. Adjust VALIDATION_START_YEAR/MONTH.')

validation_model = make_model_pipeline()
validation_model.fit(train_pairs[FEATURE_COLUMNS], np.log1p(train_pairs['target_amount']))
test_predicted = np.expm1(validation_model.predict(test_pairs[FEATURE_COLUMNS])).clip(min=0)
test_actual = test_pairs['target_amount'].to_numpy(dtype=float)
test_naive = test_pairs['present_amount'].to_numpy(dtype=float)

def regression_metrics(actual, predicted, model_name):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    nonzero = np.abs(actual) > EPSILON
    ape = np.abs((actual[nonzero] - predicted[nonzero]) / actual[nonzero])
    smape = 2 * np.abs(predicted - actual) / (np.abs(actual) + np.abs(predicted) + EPSILON)
    return {
        'model': model_name,
        'n': int(len(actual)),
        'MAE': float(mean_absolute_error(actual, predicted)),
        'RMSE': float(np.sqrt(mean_squared_error(actual, predicted))),
        'R2': float(r2_score(actual, predicted)),
        'MAPE_%': float(np.mean(ape) * 100) if len(ape) else np.nan,
        'sMAPE_%': float(np.mean(smape) * 100),
        'WAPE_%': float(np.sum(np.abs(actual - predicted)) / (np.sum(np.abs(actual)) + EPSILON) * 100),
    }

# Select the blend weight on the validation set. This lets the historical data decide
# whether pure ML, pure persistence, or a conservative combination is most accurate.
blend_rows = []
for weight in np.linspace(0.0, 1.0, 11):
    blended = (weight * test_predicted) + ((1.0 - weight) * test_naive)
    row = regression_metrics(test_actual, blended, f'Blend {weight:.2f}')
    row['blend_weight'] = float(weight)
    blend_rows.append(row)
blend_matrix = pd.DataFrame(blend_rows).set_index('model')
best_model_name = blend_matrix.sort_values(['WAPE_%', 'MAE', 'RMSE']).index[0]
BLEND_WEIGHT = float(blend_matrix.loc[best_model_name, 'blend_weight'])
selected_predicted = (BLEND_WEIGHT * test_predicted) + ((1.0 - BLEND_WEIGHT) * test_naive)
validation_matrix = blend_matrix.drop(columns=['blend_weight'])
display(validation_matrix.round(4))
print(f'Selected validation blend: {best_model_name} (BLEND_WEIGHT={BLEND_WEIGHT:.2f})')

# Refit only after validation, so the production model can use every historical pair.
production_model = make_model_pipeline()
production_model.fit(monthly_base[FEATURE_COLUMNS], np.log1p(monthly_base['target_amount']))
print('Validation and production refit completed.')


In [ ]:
def normalize_rate(value) -> float:
    rate = float(value)
    # Accept either 0.15 or 15.0 for convenience, while storing calculations as decimals.
    return rate / 100.0 if abs(rate) > 1.0 else rate

def normalized_tax_rates(rates) -> dict:
    required = set(TAX_RATES)
    missing = sorted(required - set(rates))
    if missing:
        raise ValueError(f'Missing required tax rates: {missing}')
    normalized = {key: normalize_rate(rates[key]) for key in required}
    if any(rate < 0 for rate in normalized.values()):
        raise ValueError('Tax rates cannot be negative.')
    return normalized

def rateable_values(monthly_base_amount: float, nrv_constant: float) -> dict:
    # Formula sheet, Section 2: the six base calculations in the prescribed order.
    monthly_base_amount = float(monthly_base_amount)
    annual_amount = monthly_base_amount * 12.0
    letting_value = annual_amount + (annual_amount / 3.0)
    gross_rateable_value_property = letting_value - ((letting_value * 9.0 / 10.0) * 9.0 / 10.0)
    net_rateable_value_property = gross_rateable_value_property - (gross_rateable_value_property / 10.0)
    gross_rateable_value_structure = gross_rateable_value_property - annual_amount
    net_rateable_value_structure = gross_rateable_value_structure - (gross_rateable_value_structure / 10.0)
    return {
        'AM': annual_amount,
        'LV': letting_value,
        'GRVP': gross_rateable_value_property,
        'NRVP': net_rateable_value_property,
        'GRVS': gross_rateable_value_structure,
        'NRVS': net_rateable_value_structure,
        'nrv_constant': float(nrv_constant),
    }

def dual_component_tax(am, nrvs, nrv_constant, rate) -> float:
    # Used by MECESS, Tree Cess, WBT, SBT, and EGCESS.
    return (((am / 2.0) * nrv_constant) * rate) + ((nrvs / 2.0) * rate)

def formula_taxes(monthly_base_amount: float, billing_month: int, rates: dict,
                   structure_type: str = 'other', water_tax_included: bool = True,
                   property_tax_applicable = None) -> dict:
    rates = normalized_tax_rates(rates)
    structure_key = str(structure_type).strip().lower()
    nrv_constant = NRV_CONSTANTS['mbpt'] if structure_key == 'mbpt' else NRV_CONSTANTS['other']
    bases = rateable_values(monthly_base_amount, nrv_constant)
    am, nrvp, nrvs = bases['AM'], bases['NRVP'], bases['NRVS']
    if property_tax_applicable is None:
        property_tax_applicable = structure_key == 'mbpt'
    taxes = {
        'mecess': 0.0, 'tree_cess': 0.0, 'property_tax': 0.0,
        'wbt': 0.0, 'sbt': 0.0, 'egcess': 0.0, 'street_tax': 0.0,
    }
    billing_month = int(billing_month)
    if billing_month in PRE_TAX_MONTHS:
        if property_tax_applicable:
            water_component = (nrvp * rates['water']) / 2.0 if water_tax_included else 0.0
            taxes['property_tax'] = ((nrvs * rates['general']) / 2.0) + ((nrvp * rates['sewerage']) / 2.0) + water_component
        taxes['wbt'] = dual_component_tax(am, nrvs, nrv_constant, rates['wbt'])
        taxes['sbt'] = dual_component_tax(am, nrvs, nrv_constant, rates['sbt'])
        taxes['egcess'] = dual_component_tax(am, nrvs, nrv_constant, rates['egcess'])
        taxes['street_tax'] = (nrvp * rates['street']) / 2.0
    elif billing_month in POST_TAX_MONTHS:
        taxes['mecess'] = dual_component_tax(am, nrvs, nrv_constant, rates['mecess'])
        taxes['tree_cess'] = dual_component_tax(am, nrvs, nrv_constant, rates['tree_cess'])
    taxes['total_formula_tax'] = float(sum(taxes.values()))
    return {**bases, **taxes, 'tax_schedule': 'pre_taxes' if billing_month in PRE_TAX_MONTHS else ('post_taxes' if billing_month in POST_TAX_MONTHS else 'none')}

# Formula-sheet worked example: AM=168,000, LV=224,000, GRVP=42,560, NRVP=38,304, GRVS=-125,440, NRVS=-112,896.
worked_bases = rateable_values(14_000.0, 0.837)
expected_bases = {'AM': 168_000.0, 'LV': 224_000.0, 'GRVP': 42_560.0, 'NRVP': 38_304.0, 'GRVS': -125_440.0, 'NRVS': -112_896.0}
for key, expected in expected_bases.items():
    assert np.isclose(worked_bases[key], expected), f'{key} formula mismatch'
worked_pre_tax = formula_taxes(14_000.0, 4, {**TAX_RATES}, structure_type='mbpt', water_tax_included=True)
expected_tax_values = {'property_tax': -5_785.92, 'wbt': 415.80, 'sbt': 277.20, 'egcess': 138.60, 'street_tax': 383.04}
for key, expected in expected_tax_values.items():
    assert np.isclose(worked_pre_tax[key], expected), f'{key} formula mismatch'
print('Formula unit tests passed against the worked example in Tax_Formulas_Expanded.md.')


In [ ]:
def infer_gst_rate(tax_amount: float, base_amount: float) -> float:
    if abs(float(base_amount)) <= EPSILON:
        return 0.0
    return max(0.0, float(tax_amount) / float(base_amount))

def next_year_month(year: int, month: int):
    month = int(month) + 1
    year = int(year)
    if month == 13:
        year += 1
        month = 1
    return year, month

def forecast_base_amount(model, present_bill: dict, target_year: int, target_month: int) -> float:
    current_year = int(present_bill['year'])
    current_month = int(present_bill['month'])
    target_period = period_index(target_year, target_month)
    current_period = period_index(current_year, current_month)
    if target_period <= current_period:
        raise ValueError('target_year/target_month must be after the present bill period.')
    current_amount = max(0.0, float(present_bill['amount']))
    current_cgst = max(0.0, float(present_bill.get('cgst', 0.0)))
    current_sgst = max(0.0, float(present_bill.get('sgst', 0.0)))
    current_area = float(present_bill.get('area', TRAINING_AREA_MEDIAN) or TRAINING_AREA_MEDIAN)
    current_area = current_area if current_area > 0 else TRAINING_AREA_MEDIAN
    frequency = normalize_frequency(present_bill.get('area_bill_per', 'monthly'))
    category = present_bill.get('line_category', 'rent')
    if category not in BASE_CATEGORIES:
        category = 'rent'
    cgst_rate = infer_gst_rate(current_cgst, current_amount)
    sgst_rate = infer_gst_rate(current_sgst, current_amount)
    while current_period < target_period:
        next_year, next_month = next_year_month(current_year, current_month)
        next_period = period_index(next_year, next_month)
        feature_row = pd.DataFrame([{
            'present_amount': current_amount,
            'present_cgst': current_cgst,
            'present_sgst': current_sgst,
            'present_area': current_area,
            'present_year': current_year,
            'present_month': current_month,
            'target_year': next_year,
            'target_month': next_month,
            'horizon_months': next_period - current_period,
            'present_amount_per_area': current_amount / current_area if current_area > 0 else 0.0,
            'present_log_amount': np.log1p(current_amount),
            'billing_frequency': frequency,
            'line_category': category,
        }])
        raw_prediction = max(0.0, float(np.expm1(model.predict(feature_row[FEATURE_COLUMNS]))[0]))
        current_amount = max(0.0, (BLEND_WEIGHT * raw_prediction) + ((1.0 - BLEND_WEIGHT) * current_amount))
        current_cgst = current_amount * cgst_rate
        current_sgst = current_amount * sgst_rate
        current_year, current_month, current_period = next_year, next_month, next_period
    return current_amount

def predict_billing_price(present_bill: dict, target_year: int, target_month: int,
                         model=production_model, rates=TAX_RATES, structure_type='other',
                         water_tax_included=True, target_area=None, target_area_bill_per=None) -> dict:
    required_inputs = ['year', 'month', 'amount', 'cgst', 'sgst', 'billing_charge', 'area_bill_per', 'area']
    missing = [key for key in required_inputs if key not in present_bill]
    if missing:
        raise ValueError(f'Missing frontend fields: {missing}')
    normalized_rates = normalized_tax_rates(rates)
    target_frequency = normalize_frequency(target_area_bill_per or present_bill['area_bill_per'])
    present_frequency = normalize_frequency(present_bill['area_bill_per'])
    present_area = float(present_bill['area'] or TRAINING_AREA_MEDIAN)
    present_area = present_area if present_area > 0 else TRAINING_AREA_MEDIAN
    target_area = float(target_area if target_area is not None else present_area)
    target_area = target_area if target_area > 0 else present_area
    predicted_monthly_base = forecast_base_amount(model, present_bill, target_year, target_month)
    present_period_months = BILLING_PERIOD_MONTHS[present_frequency]
    target_period_months = BILLING_PERIOD_MONTHS[target_frequency]
    billing_charge = max(0.0, float(present_bill['billing_charge'] or 0.0))
    area_charge = billing_charge * (target_area / present_area) * (target_period_months / present_period_months)
    taxable_base = predicted_monthly_base + area_charge
    cgst_rate = infer_gst_rate(present_bill['cgst'], present_bill['amount'])
    sgst_rate = infer_gst_rate(present_bill['sgst'], present_bill['amount'])
    predicted_cgst = taxable_base * cgst_rate
    predicted_sgst = taxable_base * sgst_rate
    structure_key = str(structure_type).strip().lower()
    nrv_constant = NRV_CONSTANTS['mbpt'] if structure_key == 'mbpt' else NRV_CONSTANTS['other']
    tax_breakdown = formula_taxes(
        predicted_monthly_base, int(target_month), normalized_rates,
        structure_type=structure_key, water_tax_included=water_tax_included,
    )
    final_price = taxable_base + predicted_cgst + predicted_sgst + tax_breakdown['total_formula_tax']
    return {
        'target_year': int(target_year),
        'target_month': int(target_month),
        'predicted_monthly_base_amount': predicted_monthly_base,
        'area_charge': area_charge,
        'inferred_cgst_rate': cgst_rate,
        'inferred_sgst_rate': sgst_rate,
        'predicted_cgst': predicted_cgst,
        'predicted_sgst': predicted_sgst,
        'present_area': present_area,
        'target_area': target_area,
        'present_area_bill_per': present_frequency,
        'target_area_bill_per': target_frequency,
        'area_period_months': target_period_months,
        'final_billing_price': final_price,
        **tax_breakdown,
    }

# Example frontend payload. Replace this dictionary with the actual request body.
example_present_bill = {
    'year': 2025,
    'month': 8,
    'amount': 14_000.0,
    'cgst': 1_260.0,
    'sgst': 1_260.0,
    'billing_charge': 250.0,
    'area_bill_per': 'monthly',
    'area': 1_000.0,
}
example_prediction = predict_billing_price(example_present_bill, target_year=2026, target_month=4, structure_type='mbpt')
display(pd.Series(example_prediction).to_frame('value'))


In [ ]:
# Stronger production model: XGBoost on log1p(base_amount).
# This cell intentionally mirrors train_billing_model.py and the browser artifact.
from xgboost import XGBRegressor

XGB_BASE_FEATURES = [
    'present_amount', 'present_cgst', 'present_sgst', 'present_area',
    'present_year', 'present_month', 'target_year', 'target_month',
    'horizon_months', 'present_amount_per_area', 'present_log_amount',
    'billing_frequency', 'line_category',
]

def make_xgb_features(frame: pd.DataFrame) -> pd.DataFrame:
    encoded = frame[XGB_BASE_FEATURES].copy()
    return pd.get_dummies(encoded, columns=['billing_frequency', 'line_category'], dtype=float)

def make_xgb_model() -> XGBRegressor:
    return XGBRegressor(
        n_estimators=400, max_depth=7, learning_rate=0.04, min_child_weight=5,
        subsample=0.9, colsample_bytree=0.95, reg_alpha=0.05, reg_lambda=3.0,
        objective='reg:squarederror', eval_metric='rmse', tree_method='hist',
        n_jobs=1, random_state=RANDOM_STATE,
    )

xgb_train_features = make_xgb_features(train_pairs)
xgb_test_features = make_xgb_features(test_pairs).reindex(columns=xgb_train_features.columns, fill_value=0.0)
xgb_validation_model = make_xgb_model()
xgb_validation_model.fit(
    xgb_train_features, np.log1p(train_pairs['target_amount']),
    eval_set=[(xgb_test_features, np.log1p(test_pairs['target_amount']))], verbose=False,
)
xgb_test_predicted = np.expm1(xgb_validation_model.predict(xgb_test_features)).clip(min=0)

def xgb_regression_metrics(actual, predicted, model_name):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    smape = 2 * np.abs(predicted - actual) / (np.abs(actual) + np.abs(predicted) + EPSILON)
    return {
        'model': model_name, 'n': int(len(actual)),
        'MAE': float(mean_absolute_error(actual, predicted)),
        'RMSE': float(np.sqrt(mean_squared_error(actual, predicted))),
        'R2_raw_currency': float(r2_score(actual, predicted)),
        'R2_log_space': float(r2_score(np.log1p(actual), np.log1p(predicted.clip(min=0)))),
        'sMAPE_%': float(np.mean(smape) * 100),
        'WAPE_%': float(np.sum(np.abs(actual - predicted)) / (np.sum(np.abs(actual)) + EPSILON) * 100),
    }

test_actual = test_pairs['target_amount'].to_numpy(dtype=float)
validation_matrix = pd.DataFrame([
    xgb_regression_metrics(test_actual, xgb_test_predicted, 'XGBoost log1p (selected)'),
    xgb_regression_metrics(test_actual, test_pairs['present_amount'], 'Persistence baseline'),
]).set_index('model')
selected_predicted = xgb_test_predicted
BLEND_WEIGHT = 1.0
display(validation_matrix.round(4))

# Refit on every historical pair for the frontend/production forecast.
xgb_all_features = make_xgb_features(monthly_base).reindex(columns=xgb_train_features.columns, fill_value=0.0)
production_model = make_xgb_model()
production_model.fit(xgb_all_features, np.log1p(monthly_base['target_amount']), verbose=False)
xgb_feature_columns = list(xgb_train_features.columns)

def forecast_base_amount(model, present_bill: dict, target_year: int, target_month: int) -> float:
    current_year, current_month = int(present_bill['year']), int(present_bill['month'])
    target_period = period_index(target_year, target_month)
    current_period = period_index(current_year, current_month)
    if target_period <= current_period:
        raise ValueError('target_year/target_month must be after the present bill period.')
    current_amount = max(0.0, float(present_bill['amount']))
    current_cgst = max(0.0, float(present_bill.get('cgst', 0.0)))
    current_sgst = max(0.0, float(present_bill.get('sgst', 0.0)))
    current_area = float(present_bill.get('area', TRAINING_AREA_MEDIAN) or TRAINING_AREA_MEDIAN)
    current_area = current_area if current_area > 0 else TRAINING_AREA_MEDIAN
    frequency = normalize_frequency(present_bill.get('area_bill_per', 'monthly'))
    category = present_bill.get('line_category', 'rent') if present_bill.get('line_category', 'rent') in BASE_CATEGORIES else 'rent'
    cgst_rate = infer_gst_rate(current_cgst, current_amount)
    sgst_rate = infer_gst_rate(current_sgst, current_amount)
    while current_period < target_period:
        next_year, next_month = next_year_month(current_year, current_month)
        next_period = period_index(next_year, next_month)
        feature_row = pd.DataFrame([{
            'present_amount': current_amount, 'present_cgst': current_cgst, 'present_sgst': current_sgst,
            'present_area': current_area, 'present_year': current_year, 'present_month': current_month,
            'target_year': next_year, 'target_month': next_month, 'horizon_months': next_period - current_period,
            'present_amount_per_area': current_amount / current_area if current_area > 0 else 0.0,
            'present_log_amount': np.log1p(current_amount), 'billing_frequency': frequency, 'line_category': category,
        }])
        xgb_row = make_xgb_features(feature_row).reindex(columns=xgb_feature_columns, fill_value=0.0)
        current_amount = max(0.0, float(np.expm1(model.predict(xgb_row)[0])))
        current_cgst, current_sgst = current_amount * cgst_rate, current_amount * sgst_rate
        current_year, current_month, current_period = next_year, next_month, next_period
    return current_amount

# The formula function remains unchanged; only its default model is rebound to XGBoost.
predict_billing_price.__defaults__ = (production_model, TAX_RATES, 'other', True, None, None)
example_prediction = predict_billing_price(example_present_bill, target_year=2026, target_month=4, structure_type='mbpt')
display(pd.Series(example_prediction).to_frame('value'))
print(f'XGBoost validation raw-currency R2: {validation_matrix.loc["XGBoost log1p (selected)", "R2_raw_currency"]:.4f}')
print(f'XGBoost validation log-space R2: {validation_matrix.loc["XGBoost log1p (selected)", "R2_log_space"]:.4f}')


## Validation matrix and evaluation strategy

The `validation_matrix` above is the required time-based test report. Its rows compare the hybrid model with a persistence baseline; its columns are:

- **MAE**: average absolute error in the dataset's amount units.
- **RMSE**: penalizes large billing errors more heavily.
- **R2**: variance explained; use alongside error measures because extreme values are present.
- **MAPE / sMAPE**: percentage errors; sMAPE is safer when amounts are small.
- **WAPE**: total absolute error divided by total actual amount, useful for portfolio-level accuracy.

A model is preferred when it improves MAE, RMSE, WAPE, and sMAPE versus the persistence baseline without an unacceptable R2 decline. The final formula layer is deterministic once the base amount, master rates, structure type, water-tax flag, and area-charge interpretation are supplied. The worked-example assertions verify the six intermediate rateable values and the tax equations independently of the machine-learning model.

For production use, replace the example `TAX_RATES` with the effective master-record rates for the target period and set `structure_type='mbpt'` only when the property qualifies. The notebook cannot honestly guarantee an exact rupee value when those external master inputs are absent; it does guarantee that the supplied rates are applied using the formula sheet's order and schedule.